In [ ]:
!pip install evaluate rouge_score sacrebleu bert_score scikit-learn pandas

In [3]:
!pip install --upgrade transformers tokenizers
!pip install protobuf


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


# Inference & Evaluation Matrix

In [ ]:
import os
import json
import torch
import pandas as pd
import numpy as np
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.notebook import tqdm


os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# --- 1. CONFIGURATION ---
# These files must contain the rationales generated by your Phase 3 Qwen model
TEST_1_PATH = "test_1_with_SFT_rationales.json" 
TEST_2_PATH = "test_2_with_SFT_rationales.json"
CACHE_FILE = "grounding_cache.json"

# The trained verification head from Phase 4
MODEL_PATH = "/workspace/deberta-chart-verifier-final-SFT"

# Load HuggingFace metrics
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def load_data(filepath):
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    raise FileNotFoundError(f"Missing file: {filepath}")

# --- 2. REASONING TYPE COMPRESSION ---
def compress_reasoning_type(rtype):
    if not rtype or pd.isna(rtype): return "unknown"
    rtype_lower = str(rtype).lower()
    if rtype_lower == "unknown": return "unknown"
    
    if "extremum" in rtype_lower: return "Find Extremum"
    if "compute derived value" in rtype_lower or "math" in rtype_lower: return "Compute Derived Value"
    if "compare" in rtype_lower or "filter" in rtype_lower: return "Filter & Compare"
    if "retrieve value" in rtype_lower: return "Retrieve Value"
    if "feature" in rtype_lower or "color" in rtype_lower or "caption" in rtype_lower: return "Chart Features"
    if "range" in rtype_lower: return "Determine Range"
    if "multichart" in rtype_lower or "multiaxis" in rtype_lower: return "Multi-Chart/Axis Analysis"
    if "kcs" in rtype_lower: return "Knowledge/Common Sense (KCS)"
    
    return "Other"

# --- 3. INFERENCE PIPELINE ---
def run_deberta_inference(data, cache, model, tokenizer, device):
    results = []
    for item in tqdm(data, desc="Running DeBERTa Inference"):
        img_path = item.get("local_image_path", "").replace("\\\\", "/").replace("\\", "/")
        if img_path not in cache or not cache[img_path].get("deplot_table"):
            continue
            
        desc = item.get("title_description", "")
        if desc:
            words = desc.split()
            if len(words) > 50:
                desc = " ".join(words[:50]) + "..."
                
        context_str = (
            f"Title: {desc}\n"
            f"Table: {cache[img_path]['deplot_table']}\n"
            f"Spec: {cache[img_path]['vega_lite_spec']}\n"
            f"Claim: {item['claim']}"
        )
        
        rationale = item.get("generated_rationale", "")
        
        inputs = tokenizer(
            context_str, 
            rationale, 
            truncation=True, 
            max_length=2048, 
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            prediction = torch.argmax(logits, dim=1).item()
            
        # Append the prediction back into the dictionary
        item["predicted_label"] = bool(prediction)
        results.append(item)
        
    return results

# --- 4. EVALUATION REPORTING ---
def evaluate_pipeline(test1_data, test2_data, model_name="DePlot-Qwen3B-DeBERTa"):
    y_true_1 = [1 if item["label"] else 0 for item in test1_data]
    y_pred_1 = [1 if item["predicted_label"] else 0 for item in test1_data]
    
    y_true_2 = [1 if item["label"] else 0 for item in test2_data]
    y_pred_2 = [1 if item["predicted_label"] else 0 for item in test2_data]
    
    acc_1 = accuracy_score(y_true_1, y_pred_1) * 100
    f1_1 = f1_score(y_true_1, y_pred_1, average='macro') * 100
    acc_2 = accuracy_score(y_true_2, y_pred_2) * 100
    f1_2 = f1_score(y_true_2, y_pred_2, average='macro') * 100
    avg_acc = (acc_1 + acc_2) / 2
    
    all_data = test1_data + test2_data
    preds_text = [item.get("generated_rationale", "") for item in all_data]
    refs_text = [item.get("explanation", "") for item in all_data]
    
    print("\nComputing Text Generation Metrics (This may take a minute for BERTScore)...")
    bleu_result = bleu_metric.compute(predictions=preds_text, references=refs_text)
    rouge_result = rouge_metric.compute(predictions=preds_text, references=refs_text)
    bert_result = bertscore_metric.compute(predictions=preds_text, references=refs_text, lang="en")
    
    bleu_score = bleu_result['score']
    rouge_l_score = rouge_result['rougeL'] * 100
    bert_score_avg = np.mean(bert_result['f1']) * 100

    print("\n" + "="*115)
    print(f"  CHARTCHECK EVALUATION MATRIX — {model_name}")
    print("="*115)
    print(f"{'Model':>28} {'Task':>5} {'Test1_Acc':>10} {'Test1_F1':>9} {'Test2_Acc':>10} {'Test2_F1':>9} {'Avg_Acc':>8} {'BLEU':>6} {'ROUGE-L':>8} {'BERTScore':>10}")
    
    print(f"{'Qwen2.5-VL-7B (Zero-Shot)':>28} {'M':>5} {82.3:>10.1f} {82.2:>9.1f} {85.0:>10.1f} {85.0:>9.1f} {83.7:>8.1f} {5.1:>6.1f} {32.7:>8.1f} {89.5:>10.1f}")
    
    print(f"{model_name:>28} {'M':>5} {acc_1:>10.1f} {f1_1:>9.1f} {acc_2:>10.1f} {f1_2:>9.1f} {avg_acc:>8.1f} {bleu_score:>6.1f} {rouge_l_score:>8.1f} {bert_score_avg:>10.1f}")
    print("="*115)
    
    acc_diff = avg_acc - 83.7
    sign = "+" if acc_diff >= 0 else ""
    print(f"\nAvg accuracy vs Qwen2.5-VL-7B baseline : {sign}{acc_diff:.1f}%")
    print(f"Test 1  →  Acc: {acc_1:.1f}%   Macro-F1: {f1_1:.1f}")
    print(f"Test 2  →  Acc: {acc_2:.1f}%   Macro-F1: {f1_2:.1f}")
    print(f"Average →  Acc: {avg_acc:.1f}%\n")
    
    print("Test 1 — Per-class report:")
    target_names = ['REFUTED', 'SUPPORTED']
    print(classification_report(y_true_1, y_pred_1, target_names=target_names, digits=2))

    df_test1 = pd.DataFrame(test1_data)
    df_test1['is_correct'] = df_test1['label'] == df_test1['predicted_label']
    
    print("\nFine-grained: performance by chart type (Test 1)")
    print("──────────────────────────────────────────────────")
    print(f"{'chart_type':<30} {'Accuracy':>10} {'Count':>7}\n")
    
    chart_group = df_test1.groupby('chart_type').agg(
        accuracy=('is_correct', 'mean'),
        count=('is_correct', 'count')
    ).sort_values(by='count', ascending=False)
    
    for index, row in chart_group.iterrows():
        print(f"{index:<30} {row['accuracy']*100:>9.1f}% {int(row['count']):>7}")
    print(f"  Total samples accounted for: {len(df_test1)} / {len(df_test1)}\n")
    
    print("\nFine-grained: performance by reasoning type (Test 1)")
    print("──────────────────────────────────────────────────")
    print(f"{'reasoning_type':<30} {'Accuracy':>10} {'Count':>7}\n")
    
    df_test1['compressed_reasoning'] = df_test1['reasoning_type'].apply(compress_reasoning_type)
    
    reasoning_group = df_test1.groupby('compressed_reasoning').agg(
        accuracy=('is_correct', 'mean'),
        count=('is_correct', 'count')
    ).sort_values(by='count', ascending=False)
    
    for index, row in reasoning_group.iterrows():
        print(f"{index:<30} {row['accuracy']*100:>9.1f}% {int(row['count']):>7}")
    print(f"  Total samples accounted for: {len(df_test1)} / {len(df_test1)}\n")

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("Loading Verification Head and Grounding Data...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
    model.eval()
    
    cache = load_data(CACHE_FILE)
    test1_raw = load_data(TEST_1_PATH)
    test2_raw = load_data(TEST_2_PATH)
    
    print("Processing Test 1...")
    test1_evaluated = run_deberta_inference(test1_raw, cache, model, tokenizer, device)
    
    print("Processing Test 2...")
    test2_evaluated = run_deberta_inference(test2_raw, cache, model, tokenizer, device)
    
    # Save the files with predictions appended to prevent needing to re-run inference later
    with open("test_1_evaluated.json", "w", encoding="utf-8") as f:
        json.dump(test1_evaluated, f, indent=4)
    with open("test_2_evaluated.json", "w", encoding="utf-8") as f:
        json.dump(test2_evaluated, f, indent=4)
        
    print("\nStarting Matrix Calculation...")
    evaluate_pipeline(test1_evaluated, test2_evaluated, model_name="DePlot-Qwen3B-DeBERTa")

# Evaluator without finetuned model

In [3]:
import os
import json
import torch
import pandas as pd
import numpy as np
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.notebook import tqdm

os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# --- 1. CONFIGURATION ---
TEST_1_PATH = "test_1_with_SFT_rationales.json" 
TEST_2_PATH = "test_2_with_SFT_rationales.json"

# The base off-the-shelf NLI model
MODEL_PATH = "cross-encoder/nli-deberta-v3-large"

# Load HuggingFace metrics
bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

def load_data(filepath):
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            return json.load(f)
    raise FileNotFoundError(f"Missing file: {filepath}")

# --- 2. REASONING TYPE COMPRESSION ---
def compress_reasoning_type(rtype):
    if not rtype or pd.isna(rtype): return "unknown"
    rtype_lower = str(rtype).lower()
    if rtype_lower == "unknown": return "unknown"
    
    if "extremum" in rtype_lower: return "Find Extremum"
    if "compute derived value" in rtype_lower or "math" in rtype_lower: return "Compute Derived Value"
    if "compare" in rtype_lower or "filter" in rtype_lower: return "Filter & Compare"
    if "retrieve value" in rtype_lower: return "Retrieve Value"
    if "feature" in rtype_lower or "color" in rtype_lower or "caption" in rtype_lower: return "Chart Features"
    if "range" in rtype_lower: return "Determine Range"
    if "multichart" in rtype_lower or "multiaxis" in rtype_lower: return "Multi-Chart/Axis Analysis"
    if "kcs" in rtype_lower: return "Knowledge/Common Sense (KCS)"
    
    return "Other"

# --- 3. INFERENCE PIPELINE ---
def run_nli_inference(data, model, tokenizer, device):
    results = []
    for item in tqdm(data, desc="Running Zero-Shot NLI Inference"):
        claim = item.get("claim", "")
        rationale = item.get("generated_rationale", "")
        
        # NLI setup: Premise = Rationale, Hypothesis = Claim
        inputs = tokenizer(
            text=rationale, 
            text_pair=claim, 
            truncation="only_first", # Truncates rationale if necessary, preserves claim
            max_length=512, # 512 is sufficient for purely text-based logic mapping
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            prediction = torch.argmax(logits, dim=1).item()
            
        # cross-encoder/nli-deberta-v3-large label mapping:
        # 0: contradiction, 1: entailment, 2: neutral
        # We consider Entailment (1) as True (Supported), others as False (Refuted)
        item["predicted_label"] = True if prediction == 1 else False
        results.append(item)
        
    return results

# --- 4. EVALUATION REPORTING ---
# --- 4. EVALUATION REPORTING ---
def evaluate_pipeline(test1_data, test2_data, model_name="Zero-Shot NLI (DeBERTa-Large)"):
    y_true_1 = [1 if item["label"] else 0 for item in test1_data]
    y_pred_1 = [1 if item["predicted_label"] else 0 for item in test1_data]
    
    y_true_2 = [1 if item["label"] else 0 for item in test2_data]
    y_pred_2 = [1 if item["predicted_label"] else 0 for item in test2_data]
    
    acc_1 = accuracy_score(y_true_1, y_pred_1) * 100
    f1_1 = f1_score(y_true_1, y_pred_1, average='macro') * 100
    acc_2 = accuracy_score(y_true_2, y_pred_2) * 100
    f1_2 = f1_score(y_true_2, y_pred_2, average='macro') * 100
    avg_acc = (acc_1 + acc_2) / 2
    
    all_data = test1_data + test2_data
    
    # THE FIX: Sanitize empty strings to bypass the bert_score bug
    preds_text = [item.get("generated_rationale", " ") if str(item.get("generated_rationale", "")).strip() else "." for item in all_data]
    refs_text = [item.get("explanation", " ") if str(item.get("explanation", "")).strip() else "." for item in all_data]
    
    # Check if the dataset actually has ground-truth explanations to score against
    has_refs = any(ref != "." for ref in refs_text)
    
    if has_refs:
        print("\nComputing Text Generation Metrics (This may take a minute for BERTScore)...")
        bleu_result = bleu_metric.compute(predictions=preds_text, references=refs_text)
        rouge_result = rouge_metric.compute(predictions=preds_text, references=refs_text)
        bert_result = bertscore_metric.compute(predictions=preds_text, references=refs_text, lang="en")
        
        bleu_score = bleu_result['score']
        rouge_l_score = rouge_result['rougeL'] * 100
        bert_score_avg = np.mean(bert_result['f1']) * 100
    else:
        print("\nNo ground-truth explanations found. Skipping NLP metrics.")
        bleu_score, rouge_l_score, bert_score_avg = 0.0, 0.0, 0.0

    print("\n" + "="*115)
    print(f"  CHARTCHECK EVALUATION MATRIX — {model_name}")
    print("="*115)
    print(f"{'Model':>32} {'Task':>5} {'Test1_Acc':>10} {'Test1_F1':>9} {'Test2_Acc':>10} {'Test2_F1':>9} {'Avg_Acc':>8} {'BLEU':>6} {'ROUGE-L':>8} {'BERTScore':>10}")
    
    print(f"{'Qwen2.5-VL-7B (Zero-Shot)':>32} {'M':>5} {82.3:>10.1f} {82.2:>9.1f} {85.0:>10.1f} {85.0:>9.1f} {83.7:>8.1f} {5.1:>6.1f} {32.7:>8.1f} {89.5:>10.1f}")
    
    print(f"{model_name:>32} {'M':>5} {acc_1:>10.1f} {f1_1:>9.1f} {acc_2:>10.1f} {f1_2:>9.1f} {avg_acc:>8.1f} {bleu_score:>6.1f} {rouge_l_score:>8.1f} {bert_score_avg:>10.1f}")
    print("="*115)
    
    acc_diff = avg_acc - 83.7
    sign = "+" if acc_diff >= 0 else ""
    print(f"\nAvg accuracy vs Qwen2.5-VL-7B baseline : {sign}{acc_diff:.1f}%")
    print(f"Test 1  →  Acc: {acc_1:.1f}%   Macro-F1: {f1_1:.1f}")
    print(f"Test 2  →  Acc: {acc_2:.1f}%   Macro-F1: {f1_2:.1f}")
    print(f"Average →  Acc: {avg_acc:.1f}%\n")
    
    print("Test 1 — Per-class report:")
    target_names = ['REFUTED', 'SUPPORTED']
    print(classification_report(y_true_1, y_pred_1, target_names=target_names, digits=2))

    df_test1 = pd.DataFrame(test1_data)
    df_test1['is_correct'] = df_test1['label'] == df_test1['predicted_label']
    
    print("\nFine-grained: performance by chart type (Test 1)")
    print("──────────────────────────────────────────────────")
    print(f"{'chart_type':<30} {'Accuracy':>10} {'Count':>7}\n")
    
    chart_group = df_test1.groupby('chart_type').agg(
        accuracy=('is_correct', 'mean'),
        count=('is_correct', 'count')
    ).sort_values(by='count', ascending=False)
    
    for index, row in chart_group.iterrows():
        print(f"{index:<30} {row['accuracy']*100:>9.1f}% {int(row['count']):>7}")
    print(f"  Total samples accounted for: {len(df_test1)} / {len(df_test1)}\n")
    
    print("\nFine-grained: performance by reasoning type (Test 1)")
    print("──────────────────────────────────────────────────")
    print(f"{'reasoning_type':<30} {'Accuracy':>10} {'Count':>7}\n")
    
    df_test1['compressed_reasoning'] = df_test1['reasoning_type'].apply(compress_reasoning_type)
    
    reasoning_group = df_test1.groupby('compressed_reasoning').agg(
        accuracy=('is_correct', 'mean'),
        count=('is_correct', 'count')
    ).sort_values(by='count', ascending=False)
    
    for index, row in reasoning_group.iterrows():
        print(f"{index:<30} {row['accuracy']*100:>9.1f}% {int(row['count']):>7}")
    print(f"  Total samples accounted for: {len(df_test1)} / {len(df_test1)}\n")

# --- 5. EXECUTION ---
if __name__ == "__main__":
    print("Loading Base NLI Model...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
    model.eval()
    
    test1_raw = load_data(TEST_1_PATH)
    test2_raw = load_data(TEST_2_PATH)
    
    print("Processing Test 1...")
    test1_evaluated = run_nli_inference(test1_raw, model, tokenizer, device)
    
    print("Processing Test 2...")
    test2_evaluated = run_nli_inference(test2_raw, model, tokenizer, device)
    
    with open("test_1_evaluated_zeroshot.json", "w", encoding="utf-8") as f:
        json.dump(test1_evaluated, f, indent=4)
    with open("test_2_evaluated_zeroshot.json", "w", encoding="utf-8") as f:
        json.dump(test2_evaluated, f, indent=4)
        
    print("\nStarting Matrix Calculation...")
    evaluate_pipeline(test1_evaluated, test2_evaluated, model_name="Zero-Shot NLI-DeBERTa-Large")

Loading Base NLI Model...


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing Test 1...


Running Zero-Shot NLI Inference:   0%|          | 0/939 [00:00<?, ?it/s]

...


Starting Matrix Calculation...

Computing Text Generation Metrics (This may take a minute for BERTScore)...


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



  CHARTCHECK EVALUATION MATRIX — Zero-Shot NLI-DeBERTa-Large
                           Model  Task  Test1_Acc  Test1_F1  Test2_Acc  Test2_F1  Avg_Acc   BLEU  ROUGE-L  BERTScore
       Qwen2.5-VL-7B (Zero-Shot)     M       82.3      82.2       85.0      85.0     83.7    5.1     32.7       89.5
     Zero-Shot NLI-DeBERTa-Large     M       55.7      54.1       55.5      54.2     55.6   13.7     33.3       89.7

Avg accuracy vs Qwen2.5-VL-7B baseline : -28.1%
Test 1  →  Acc: 55.7%   Macro-F1: 54.1
Test 2  →  Acc: 55.5%   Macro-F1: 54.2
Average →  Acc: 55.6%

Test 1 — Per-class report:
              precision    recall  f1-score   support

     REFUTED       0.53      0.77      0.63       452
   SUPPORTED       0.63      0.36      0.46       487

    accuracy                           0.56       939
   macro avg       0.58      0.56      0.54       939
weighted avg       0.58      0.56      0.54       939


Fine-grained: performance by chart type (Test 1)
─────────────────────────────────

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()